# Lab 3: Word2Vec Skip-Gram and Document Embeddings

This notebook implements Word2Vec. Each document is converted to one vector by averaging the vectors of the words in its `text` column.

The model uses `vector_size=300` and `window=7`. Negative sampling keeps the manual training loop practical for this 46k-document corpus while retaining the skip-gram idea from the lab document.

In [1]:
from collections import Counter
from pathlib import Path
import random

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
INPUT_PATH = PROJECT_ROOT / "data" / "processed_data" / "preprocessed_research_papers.csv"
WORD_VECTOR_PATH = PROJECT_ROOT / "data" / "processed_data" / "word_vectors_improved.dat"
VOCAB_PATH = PROJECT_ROOT / "data" / "processed_data" / "word2vec_vocabulary.csv"
DOCUMENT_VECTOR_PATH = PROJECT_ROOT / "data" / "processed_data" / "document_vectors_improved.npy"
EMBEDDED_DATASET_PATH = PROJECT_ROOT / "data" / "processed_data" / "preprocessed_research_papers_with_embeddings_improved.csv"

VECTOR_SIZE = 300
WINDOW = 7
MIN_COUNT = 1
NEGATIVE_SAMPLES = 2
EPOCHS = 5
MAX_TRAINING_PAIRS = 1_000_000
LEARNING_RATE = 0.025
SEED = 42

papers = pd.read_csv(INPUT_PATH)
assert "text" in papers.columns
print("papers =", len(papers))
print("input =", INPUT_PATH)

papers = 46344
input = D:\4-1\NLP_Lab\NLP-Based-Research-Paper-Intelligence-System\data\processed_data\preprocessed_research_papers.csv


## 1. Build the vocabulary

A word receives an integer id. The two model matrices are initialized manually: `W_in` stores center-word vectors and `W_out` stores context-word vectors.

In [2]:
tokenized_documents = [str(text).split() for text in papers["text"].fillna("")]
word_counts = Counter(token for document in tokenized_documents for token in document)
vocabulary = sorted(word for word, count in word_counts.items() if count >= MIN_COUNT)
word_to_id = {word: index for index, word in enumerate(vocabulary)}
id_to_word = {index: word for word, index in word_to_id.items()}

document_ids = [[word_to_id[word] for word in document if word in word_to_id] for document in tokenized_documents]
vocabulary_size = len(vocabulary)
print("vocabulary size =", vocabulary_size)
print("example ids =", list(word_to_id.items())[:10])

vocabulary size = 218928
example ids = [('0', 0), ("0's", 1), ('0-0', 2), ('0-1', 3), ('0-10', 4), ('0-100', 5), ('0-10000lx', 6), ('0-100ma', 7), ('0-100v', 8), ('0-10v', 9)]


## 2. Generate skip-gram training pairs

For each center word, words within seven positions become positive context examples. Negative examples are sampled from the vocabulary and are trained toward zero.

In [3]:
def make_training_pairs(documents, window, maximum_pairs, seed):
    random_generator = random.Random(seed)
    pairs = []
    for document in documents:
        for center_position, center_id in enumerate(document):
            start = max(0, center_position - window)
            stop = min(len(document), center_position + window + 1)
            context_ids = [document[position] for position in range(start, stop) if position != center_position]
            for context_id in context_ids:
                pairs.append((center_id, context_id))
                if len(pairs) >= maximum_pairs:
                    random_generator.shuffle(pairs)
                    return pairs
    random_generator.shuffle(pairs)
    return pairs

training_pairs = make_training_pairs(document_ids, WINDOW, MAX_TRAINING_PAIRS, SEED)
print("training pairs =", len(training_pairs))
assert training_pairs

training pairs = 1000000


## 3. Train skip-gram 

This is the learning loop. The one-hot multiplication is replaced by direct row lookup: `W_in[center_id]`. The output error is also updated directly at the selected context id.

In [4]:
def sigmoid(value):
    value = np.clip(value, -15, 15)
    return 1.0 / (1.0 + np.exp(-value))


def open_matrix(path, shape, random_generator=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    expected_bytes = int(np.prod(shape)) * np.dtype("float32").itemsize
    # A supplied generator means this matrix must be freshly initialized.
    mode = "w+" if random_generator is not None else (
        "r+" if path.exists() and path.stat().st_size == expected_bytes else "w+"
    )
    matrix = np.memmap(path, mode=mode, dtype="float32", shape=shape)
    if mode == "w+" and random_generator is not None:
        for start in range(0, shape[0], 10_000):
            stop = min(start + 10_000, shape[0])
            matrix[start:stop] = random_generator.normal(
                0, 0.5 / shape[1], (stop - start, shape[1])
            ).astype(np.float32)
        matrix.flush()
    elif mode == "w+":
        matrix[:] = 0.0
        matrix.flush()
    return matrix


def train_skipgram(pairs, vocabulary_size, vector_size, negative_samples, epochs, learning_rate, seed, vector_path):
    random_generator = np.random.default_rng(seed)
    matrix_shape = (vocabulary_size, vector_size)
    W_in = open_matrix(vector_path, matrix_shape, random_generator)
    # Initialize output vectors randomly too; zero initialization weakens learning.
    W_out = open_matrix(str(vector_path) + ".out", matrix_shape, random_generator)
    losses = []

    for epoch in range(epochs):
        total_loss = 0.0
        for center_id, context_id in pairs:
            center_vector = W_in[center_id].copy()
            positive_score = sigmoid(np.dot(center_vector, W_out[context_id]))
            positive_error = positive_score - 1.0
            W_in[center_id] -= learning_rate * positive_error * W_out[context_id]
            W_out[context_id] -= learning_rate * positive_error * center_vector
            total_loss -= np.log(max(positive_score, 1e-12))

            for _ in range(negative_samples):
                negative_id = int(random_generator.integers(vocabulary_size))
                negative_score = sigmoid(np.dot(center_vector, W_out[negative_id]))
                negative_error = negative_score
                W_in[center_id] -= learning_rate * negative_error * W_out[negative_id]
                W_out[negative_id] -= learning_rate * negative_error * center_vector
                total_loss -= np.log(max(1.0 - negative_score, 1e-12))
        losses.append(total_loss / len(pairs))
        print(f"epoch {epoch + 1}: average loss = {losses[-1]:.4f}")

    W_in.flush()
    W_out.flush()
    return W_in, losses


word_vectors, losses = train_skipgram(
    training_pairs, vocabulary_size, VECTOR_SIZE, NEGATIVE_SAMPLES,
    EPOCHS, LEARNING_RATE, SEED, WORD_VECTOR_PATH
)
print("word vector matrix shape =", word_vectors.shape)
assert word_vectors.shape == (vocabulary_size, VECTOR_SIZE)

epoch 1: average loss = 2.0227
epoch 2: average loss = 1.4948
epoch 3: average loss = 1.0486
epoch 4: average loss = 0.7772
epoch 5: average loss = 0.6280
word vector matrix shape = (218928, 300)


## 4. Average word vectors into one vector per document

The document vector is the arithmetic mean of the available word vectors. Empty documents receive a zero vector.

In [5]:
def average_document_vectors(documents, word_vectors, vector_size):
    document_vectors = np.zeros((len(documents), vector_size), dtype=np.float32)
    for row_number, document in enumerate(documents):
        if document:
            document_vectors[row_number] = word_vectors[document].mean(axis=0)
    return document_vectors


document_vectors = average_document_vectors(document_ids, word_vectors, VECTOR_SIZE)
np.save(DOCUMENT_VECTOR_PATH, document_vectors)
pd.DataFrame({"word": vocabulary, "word_id": range(vocabulary_size)}).to_csv(VOCAB_PATH, index=False)

embedded_papers = papers.copy()
embedded_papers["document_vector"] = [vector.tolist() for vector in document_vectors]
try:
    embedded_papers.to_csv(EMBEDDED_DATASET_PATH, index=False)
except PermissionError:
    fallback_path = EMBEDDED_DATASET_PATH.with_name(
        EMBEDDED_DATASET_PATH.stem + "_new.csv"
    )
    embedded_papers.to_csv(fallback_path, index=False)
    EMBEDDED_DATASET_PATH = fallback_path
    print("Original embeddings CSV is locked; saved a new copy instead.")

print("document vector matrix shape =", document_vectors.shape)
print("saved dataset =", EMBEDDED_DATASET_PATH)
assert document_vectors.shape == (len(papers), VECTOR_SIZE)
assert len(embedded_papers) == len(papers)


document vector matrix shape = (46344, 300)
saved dataset = D:\4-1\NLP_Lab\NLP-Based-Research-Paper-Intelligence-System\data\processed_data\preprocessed_research_papers_with_embeddings_improved.csv


## Verification

The checks confirm that each vocabulary word has a 300-dimensional vector and each paper has exactly one averaged document vector.

In [6]:
saved_vectors = np.load(DOCUMENT_VECTOR_PATH)
assert saved_vectors.shape == (46_344, 300)
assert np.isfinite(saved_vectors).all()
assert all(len(vector) == 300 for vector in embedded_papers["document_vector"].head(10))
print("Verification passed.")
print("word vectors:", word_vectors.shape)
print("document vectors:", saved_vectors.shape)

Verification passed.
word vectors: (218928, 300)
document vectors: (46344, 300)
